### Group 8

In [ ]:
%pip install kagglehub
%pip install torch
%pip install torchvision
%pip install scikit-learn
%pip install imblearn
%pip install "ray[tune]" torchvision


In [1]:
import kagglehub

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE  # Import function for handling imbalanced data

import ray
from ray import tune
from ray.tune import Checkpoint
from ray.tune.schedulers import ASHAScheduler

import numpy as np
import pandas as pd
import os


/home/xjv5nf/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-11-01 19:27:34,275	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-11-01 19:27:34,847	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# Set seed for notebook
SEED = 47

# Get device for training
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Download latest version
path = kagglehub.dataset_download("kamilpytlak/personal-key-indicators-of-heart-disease")

df = pd.read_csv(path + "/2022/heart_2022_no_nans.csv")
df.head(10)


Using cuda device


,State,Sex,GeneralHealth,PhysicalHealthDays,MentalHealthDays,LastCheckupTime,PhysicalActivities,SleepHours,RemovedTeeth,HadHeartAttack,...,HeightInMeters,WeightInKilograms,BMI,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos
0,Alabama,Female,Very good,4.0,0.0,Within past year (anytime less than 12 months ...,Yes,9.0,None of them,No,...,1.60,71.67,27.99,No,No,Yes,Yes,"Yes, received Tdap",No,No
1,Alabama,Male,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,6.0,None of them,No,...,1.78,95.25,30.13,No,No,Yes,Yes,"Yes, received tetanus shot but not sure what type",No,No
2,Alabama,Male,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,No,8.0,"6 or more, but not all",No,...,1.85,108.86,31.66,Yes,No,No,Yes,"No, did not receive any tetanus shot in the pa...",No,Yes
3,Alabama,Female,Fair,5.0,0.0,Within past year (anytime less than 12 months ...,Yes,9.0,None of them,No,...,1.70,90.72,31.32,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,Yes
4,Alabama,Female,Good,3.0,15.0,Within past year (anytime less than 12 months ...,Yes,5.0,1 to 5,No,...,1.55,79.38,33.07,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No
5,Alabama,Male,Good,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,7.0,None of them,No,...,1.85,120.20,34.96,Yes,Yes,Yes,No,"Yes, received tetanus shot but not sure what type",No,No
6,Alabama,Female,Good,3.0,0.0,Within past year (anytime less than 12 months ...,Yes,8.0,"6 or more, but not all",No,...,1.63,88.00,33.30,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No
7,Alabama,Male,Fair,5.0,0.0,Within past year (anytime less than 12 months ...,Yes,8.0,1 to 5,Yes,...,1.75,74.84,24.37,No,Yes,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,Yes
8,Alabama,Male,Good,2.0,0.0,5 or more years ago,No,6.0,None of them,No,...,1.70,78.02,26.94,No,No,No,No,"No, did not receive any tetanus shot in the pa...",No,Yes
9,Alabama,Female,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,7.0,None of them,No,...,1.68,63.50,22.60,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No


## Model 1: Neural Network for Predicting Heart Attacks

In [3]:
# Turn categorical data in true/false columns
df = pd.get_dummies(df, drop_first=True)

In [4]:
# Split the data into train, test, validation
target_name = "HadHeartAttack_Yes"
feature_names = [c for c in df.columns if c != target_name]

X = df[feature_names].to_numpy(dtype=np.float32)
y = df[target_name].to_numpy(dtype=np.float32)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=.2,
                                                        stratify=y,
                                                        random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, 
                                                    test_size=.5,
                                                    stratify=y_temp,
                                                    random_state=SEED)


In [5]:
smote = SMOTE(random_state=SEED)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [6]:
# Train the scaler on the training data and apply it to train, validation, and
# test data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


In [7]:
# Define this problems Dataset
class HeartAttackDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features)
        self.labels = torch.tensor(labels).unsqueeze(1)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

    def __len__(self):
        return len(self.labels)

In [8]:
# Create Datasets and Dataloaders for train/val/test data
train_dataset = HeartAttackDataset(X_train, y_train)
val_dataset = HeartAttackDataset(X_val, y_val)
test_dataset = HeartAttackDataset(X_test, y_test)

In [9]:
criterion = nn.BCEWithLogitsLoss()

In [25]:
# Train the model and score it against the validation data
def train_neural_net(config, train_dataset, val_dataset):
    # Define the model, loss function, and optimizer
    model = nn.Sequential(
                nn.Linear(train_dataset.features.shape[1], config["hidden_dim1"]),
                nn.ReLU(),
                nn.Linear(config["hidden_dim1"], config["hidden_dim2"]),
                nn.ReLU(),
                nn.Linear(config["hidden_dim2"], 1)).to(device)

    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    
    num_epochs = 10
    train_dataloader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=config["batch_size"])
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
    
        for batch_idx, (inputs, targets) in enumerate(train_dataloader):
            inputs = inputs.to(config["device"])
            targets = targets.to(config["device"])
    
            # Reset gradients
            optimizer.zero_grad()
    
            # Get the model's predictions and calculate the loss
            outputs = model(inputs)
            loss = criterion(outputs, targets)
    
            # 4. Backward pass: compute gradient of the loss with respect to model parameters
            loss.backward()
    
            # 5. Optimization step: update parameters based on gradients
            optimizer.step()
    
            train_loss += loss.item()
    
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for inputs, targets in val_dataloader:
                inputs, targets = inputs.to(device), targets.to(device)
    
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
    
                # outputs are raw logits; apply sigmoid to get probabilities [0, 1]
                probs = torch.sigmoid(outputs)
                preds = (probs >= 0.5).int()
    
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())
    
        train_loss = train_loss / len(train_dataloader)
        val_loss = val_loss / len(val_dataloader)
        val_acc = accuracy_score(all_targets, all_preds)
        val_recall = recall_score(all_targets, all_preds, zero_division=0)
        val_precision = precision_score(all_targets, all_preds, zero_division=0)
        val_f1 = f1_score(all_targets, all_preds, zero_division=0)

        tune.report({
            "loss": val_loss,
            "accuracy": val_acc,
            "recall": val_recall,
            "precision": val_precision,
            "f1": val_f1
        })

In [26]:
num_trials = 10
max_num_epochs = 10
gpus_per_trial = 1
cpus_per_trial = 2

config = {
    "lr": tune.loguniform(1e-4, 1e-1), 
    "batch_size": tune.choice([16, 32, 64]),
    "hidden_dim1": tune.choice([16, 32, 64, 128]),
    "hidden_dim2": tune.choice([16, 32, 64, 128]),
    "device": device,
}

scheduler = ASHAScheduler(
    max_t=max_num_epochs,
    grace_period=1,
    reduction_factor=2,
)

train_dataset_ref = ray.put(train_dataset)
val_dataset_ref = ray.put(val_dataset)

tuner = tune.Tuner(
    tune.with_parameters(
        train_neural_net, train_dataset=train_dataset_ref, val_dataset=val_dataset_ref,
    ),
    tune_config=tune.TuneConfig(
        metric="recall",
        mode="max",
        scheduler=scheduler,
        num_samples=num_trials,
    ),
    param_space = config,
)

results = tuner.fit()

2026-11-01 20:09:40,644	INFO tune.py:1007 -- Wrote the latest version of all result files and experiment state to '/home/xjv5nf/ray_results/train_neural_net_2026-11-01_20-02-07' in 0.0056s.
2026-11-01 20:09:40,650	INFO tune.py:1039 -- Total run time: 452.89 seconds (452.87 seconds for the tuning loop).


In [27]:
best = results.get_best_result(metric="recall", mode="max")
print(best.config)
print(best.metrics["recall"])

{'lr': 0.0006874303702528443, 'batch_size': 32, 'hidden_dim1': 32, 'hidden_dim2': 64, 'device': 'cuda'}
0.5435591958302308
